In [ ]:
import numpy as np
from subprocess import PIPE, run
import matplotlib.pyplot as plt
import os
import textwrap
from waxx.control import ethernet_relay

class ExptBuilder():
    def __init__(self):
        self.__code_path__ = os.environ.get('code')
        self.__temp_exp_path__ = os.path.join(self.__code_path__, "k-exp", "kexp", "experiments", "ml_expt.py")

    def run_expt(self):
        expt_path = self.__temp_exp_path__
        run_expt_command = r"%kpy% & ar " + expt_path
        result = run(run_expt_command, stdout=PIPE, stderr=PIPE, universal_newlines=True, shell=True)
        print(result.returncode, result.stdout, result.stderr)
        os.remove(self.__temp_exp_path__)
        return result.returncode
    
    def write_experiment_to_file(self, program):
        with open(self.__temp_exp_path__, 'w') as file:
            file.write(program)
    
    def fringe_scan_expt(self, compression_point):
        script = textwrap.dedent(f"""
        from artiq.experiment import *
        from artiq.experiment import delay
        from kexp import Base, img_types, cameras
        import numpy as np
        from kexp.calibrations.tweezer import tweezer_vpd1_to_vpd2
        from kexp.calibrations.imaging import high_field_imaging_detuning
        from artiq.coredevice.sampler import Sampler
        from artiq.language import now_mu

        class hf_raman(EnvExperiment, Base):

            def prepare(self):
                Base.__init__(self,setup_camera=True,
                            camera_select=cameras.andor,
                            save_data=True,
                            imaging_type=img_types.ABSORPTION)

                self.xvar('with_imaging', [0,1])
                self.xvar('relative_phase', np.linspace(0., 2*np.pi, 15))

                self.p.v_pd_hf_tweezer_squeeze_power = {compression_point}
                self.p.t_tweezer_squeezer_ramp_2 = 29.e-3
                
                self.p.t_ramsey = 5.e-6
                self.p.t_raman_pulse = self.p.t_raman_pi_pulse / 2

                self.p.amp_imaging = .3
                self.p.t_tweezer_hold = 15.e-3
                self.p.t_tof = 80.e-6
                self.p.t_mot_load = 1.
                self.p.N_repeats = 5

                self.finish_prepare(shuffle=True)

            @kernel
            def scan_kernel(self):

                self.set_imaging_detuning(frequency_detuned=self.p.frequency_detuned_hf_midpoint)
                # self.slm.write_phase_mask_kernel(phase=self.p.phase_slm_mask)
                self.imaging.set_power(self.p.amp_imaging)

                self.prepare_hf_tweezers(ramp_down_painting=True,squeeze=True)
                self.prep_raman()

                self.raman.set(t_phase_origin_mu=now_mu())

                self.raman.pulse(self.p.t_raman_pulse)

                if self.p.with_imaging:
                    self.imaging.on()
                    delay(self.p.t_ramsey)
                    self.imaging.off()
                else:
                    delay(self.p.t_ramsey)

                self.raman.set(relative_phase=self.p.relative_phase)
                self.raman.pulse(self.p.t_raman_pulse)

                self.ttl.raman_shutter.off()

                self.set_imaging_detuning(frequency_detuned = self.p.frequency_detuned_hf_f1m1)
                self.imaging.set_power(.2)

                delay(self.p.t_tweezer_hold)
                self.tweezer.off()

                delay(self.p.t_tof)

                self.abs_image()

            @kernel
            def run(self):
                self.init_kernel(setup_slm=True)
                self.load_2D_mot(self.p.t_2D_mot_load_delay)
                self.scan()
                
            def analyze(self):
                import os
                expt_filepath = os.path.abspath(__file__)
                self.end(expt_filepath)
                """)
        return script

In [5]:
eBuilder = ExptBuilder()

In [ ]:
compression_point = np.linspace(.16,3.94,10)

for f in range(len(compression_point)):
    print('amp_img = ', compression_point[f])
    eBuilder.write_experiment_to_file(eBuilder.fringe_scan_expt(img_amp=compression_point[f]))
    eBuilder.run_expt()

0.2 119497479.0
0  20 values of dummy. 20 total shots. 60 total images expected.
Run ID: 69428
Acknowledged camera ready signal.
Camera is ready.

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 0.20499999999999996, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.20499999999999996 pi, x-center = 994, y-center = 820

 Run ID: 69428

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 0.20499999999999996, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.20499999999999996 pi, x-center = 994, y-center = 820

shot 1/20 done

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 0.20499999999999996, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.20499999999999996 pi, x-center = 994, y-center = 820

shot 2/20 done

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 0.20499999999999996, 'dimension': 